# Balanced steady-state free precession (bSSFP)

**bSSFP** is a gradient-echo sequence run with a short, constant TR and with **every gradient
axis returned to zero net area over each RF-to-RF interval**. A slice-selective pulse is played,
the slice selection is wound and unwound, one line of k-space is read out, and the phase encoding
is rewound — and the next pulse arrives before the transverse magnetisation has decayed. Because
nothing is spoiled, the magnetisation that survives one repetition is carried into the next, and
after enough repetitions the train settles into a steady state whose signal is unusually high and
whose contrast goes roughly as $\sqrt{T_2/T_1}$.

The sequence is used wherever that combination pays: cardiac cine imaging, where the signal is
strong and the acquisition is fast; MR angiography without contrast; and fetal and abdominal
imaging, where speed and motion tolerance matter more than fine $T_1$ weighting.

Two properties define it, and they are not the same property:

```text
balanced       zero net gradient area on each axis over the RF-to-RF interval
               -- a property of the waveform
steady state   the magnetisation has settled into the repeating pattern the train drives it to
               -- a property of the train and of the tissue
```

A balanced waveform is balanced from the very first repetition. A steady state takes time to
develop, and getting there without a violent transient is the job of a **start-up** or
catalyzation schedule. This notebook builds the waveform and composes the schedule; the
simulation notebook beside it watches the magnetisation actually settle.

## What this notebook builds

`sc.modules.bSSFP2DTR` is one balanced repetition. The acquisition around it — how many start-up
repetitions, in what order the lines are played, whether k-space is segmented — is ordinary
Python in this notebook.

```text
bSSFP2DTR owns          the repetition physics
                        RF-to-RF balance on every axis
                        TE and TR timing
                        the RF and receiver phase placement

this notebook owns      the start-up schedule
                        ky ordering
                        segmentation
                        interruption and restart policy
```

In [ ]:
from pathlib import Path

import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=24, grad_unit='mT/m',
    max_slew=120, slew_unit='T/m/s',
    B0=3.0,
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
)

FOV_MM = 250.0                      # square field of view
MATRIX = (64, 64)                   # readout samples, phase-encode lines
THICKNESS_MM = 6.0
FLIP_DEG = 40.0                     # bSSFP runs hot: the steady state wants a large angle
BANDWIDTH_HZ_PX = 700.0             # wide, because a short TR is the whole point

NX, NY = MATRIX
SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)

print(f'{NX} x {NY} over {FOV_MM:.0f} mm -> in-plane voxel '
      f'{FOV_MM / NX:.2f} x {FOV_MM / NY:.2f} mm, slice {THICKNESS_MM:.1f} mm')

## One repetition

The kernel takes the protocol and nothing about the acquisition. Note what it is **not** given:
no number of start-up repetitions, no line ordering, no segment count.

In [ ]:
tr = sc.modules.bSSFP2DTR(
    opts=opts, fov_mm=FOV_MM, matrix=MATRIX, thickness_mm=THICKNESS_MM,
    flip_deg=FLIP_DEG, bandwidth_hz_px=BANDWIDTH_HZ_PX,
)

print(f'TE  {tr.te_s * 1e3:.3f} ms          TR  {tr.tr_s * 1e3:.3f} ms')
print(f'TR/2 {tr.tr_s / 2 * 1e3:.3f} ms     achieved TE - TR/2  '
      f'{tr.symmetry_residual_s * 1e6:+.1f} us')
print(f'shortest symmetric TR {tr.min_tr_s * 1e3:.3f} ms, '
      f'earliest reachable echo {tr.min_te_s * 1e3:.3f} ms')

### `TE = TR/2` is the default, and it is a realisation rather than a definition

Passing neither `te_s` nor `tr_s` asks for the shortest **symmetric** repetition: the echo at the
midpoint between two pulses. That is the canonical bSSFP realisation, the one the
$\sqrt{T_2/T_1}$ contrast is quoted for, and the arrangement in which off-resonance phase accrued
in the first half of TR is unwound in the second.

It is deliberately **not** what `te_s=None` means on a spoiled gradient echo, where it asks for
the earliest reachable echo. Here that earliest echo is `min_te_s` above, and a repetition built
around it would be a legal but asymmetric bSSFP that nobody asked for.

The echo is the ADC sample at which $k=0$, and that sample sits half a dwell off the centre of
the readout window, while the delays that place it move in whole gradient rasters. For a given
readout geometry the exact midpoint may therefore not lie on the available timing lattice; the
default takes the nearest symmetric realisation and reports what it achieved in
`symmetry_residual_s`, rather than rounding the declared TE to `TR/2`.

**Asymmetry is still balanced.** Asking for an explicit `te_s` moves the echo and changes the
off-resonance behaviour; it does not stop the sequence being bSSFP, because balance is the
zero-area condition and nothing else.

In [ ]:
asymmetric = sc.modules.bSSFP2DTR(
    opts=opts, fov_mm=FOV_MM, matrix=MATRIX, thickness_mm=THICKNESS_MM,
    flip_deg=FLIP_DEG, bandwidth_hz_px=BANDWIDTH_HZ_PX,
    te_s=tr.min_te_s, tr_s=tr.tr_s,
)

print(f'{"":12}{"TE (ms)":>10}{"TR (ms)":>10}{"TE/TR":>9}')
for name, rep in (('symmetric', tr), ('asymmetric', asymmetric)):
    print(f'{name:12}{rep.te_s * 1e3:10.3f}{rep.tr_s * 1e3:10.3f}'
          f'{rep.te_s / rep.tr_s:9.3f}')

## The timing origin is the RF effective centre

Every time this module declares is measured from the pulse's **effective centre** — the instant
the magnetisation is actually tipped — not from the start of the RF event and not from the start
of the block that holds it:

```text
TE                = echo - RF effective centre of repetition n
TR                = RF effective centre of n+1 - RF effective centre of n
balance interval  = [RF effective centre of n, RF effective centre of n+1]
```

For a symmetric sinc the effective centre is halfway through the envelope, but for a
minimum-phase pulse it is nowhere near the middle, so it is read from the waveform rather than
computed from a duration. `time_to_rf_center()` reports where it lands inside the returned block.

In [ ]:
print(f'block starts at                 0.000 ms')
print(f'RF effective centre at    {tr.time_to_rf_center() * 1e3:9.3f} ms')
print(f'echo at                   {tr.time_to_echo() * 1e3:9.3f} ms')
print(f'block ends at             {tr(line=0).duration * 1e3:9.3f} ms')
print()
print('time_to_echo() == time_to_rf_center() + te_s :',
      abs(tr.time_to_echo() - (tr.time_to_rf_center() + tr.te_s)) < 1e-12)

## What balance looks like on the three axes

Each axis returns to zero net area over the interval, and each does it differently:

```text
z   the selection gradient, with a winder before the pulse and a rephaser after it
y   the phase-encode blip, and its rewind
x   the readout prephaser, the readout lobe, and a balancing lobe after the echo
```

The z axis is the one worth looking at twice. A spoiled gradient echo needs only to unwind the
*second* half of the selection gradient — the half that acts on transverse magnetisation after it
exists. A balanced repetition also has to account for the **first** half, because the RF-to-RF
interval of the *previous* repetition runs up to this pulse's centre and that half falls inside
it. So there are two z lobes, one either side of the pulse, each carrying one half:

```text
A_post(n) + R_tail(n)  +  W_lead(n+1) + A_pre(n+1)  =  0
\____________________/    \____________________/
   played by n                played by n+1
```

Each balancing lobe cancels one selection half. Measured on the emitted waveform below.

In [ ]:
def rf_centres(seq):
    """Absolute time of every RF effective centre in a compiled sequence, seconds."""
    out, t = [], 0.0
    for i in range(1, len(seq.block_events) + 1):
        block = seq.get_block(i)
        if getattr(block, 'rf', None) is not None:
            out.append(t + float(block.rf.delay) + float(pp.calc_rf_center(block.rf)[0]))
        t += float(seq.block_durations[i])
    return np.array(out)


def m0_between(seq, a, b):
    """(M0x, M0y, M0z) over [a, b], 1/m, integrated from the emitted waveforms."""
    waveforms = seq.waveforms_and_times()[0]
    out = []
    for axis in range(3):
        t, g = waveforms[axis][0], waveforms[axis][1]
        if len(t) == 0:
            out.append(0.0)
            continue
        grid = np.unique(np.concatenate([t[(t > a) & (t < b)], [a, b]]))
        out.append(float(np.trapezoid(np.interp(grid, t, g, left=0.0, right=0.0), grid)))
    return np.array(out)


probe = sc.LogicBlock('probe')
for n in range(3):
    probe.add(n * tr.tr_s, tr(line=20 + n, phase_deg=180.0 * (n % 2)))
probe_seq = sc.compile(probe, opts)
centres = rf_centres(probe_seq)

print(f'{"interval":>10}  {"duration (ms)":>14}  {"M0x":>11}  {"M0y":>11}  {"M0z":>11}  (1/m)')
for n in range(len(centres) - 1):
    m0 = m0_between(probe_seq, centres[n], centres[n + 1])
    print(f'{n} -> {n + 1:<5}  {(centres[n + 1] - centres[n]) * 1e3:14.4f}  '
          f'{m0[0]:11.2e}  {m0[1]:11.2e}  {m0[2]:11.2e}')
print(f'\ndeclared TR {tr.tr_s * 1e3:.4f} ms')

The three lines differ in `ky`, so the y rewind is doing real work rather than returning a blip
that was never played. Each interval lasts exactly TR and each carries no net area on any axis.

### The gradient waveform, over one repetition

In [ ]:
import matplotlib.pyplot as plt

waveforms = probe_seq.waveforms_and_times()[0]
a, b = centres[0], centres[1]
fig, axes = plt.subplots(3, 1, figsize=(10, 5.4), sharex=True)
for axis, (ax, label) in enumerate(zip(axes, ('Gx', 'Gy', 'Gz'))):
    t, g = waveforms[axis][0], waveforms[axis][1]
    inside = (t >= a - 0.3e-3) & (t <= b + 0.3e-3)
    ax.plot((t[inside] - a) * 1e3, g[inside] / 1e3, lw=1.4)
    ax.axhline(0.0, color='0.8', lw=0.8)
    ax.set_ylabel(f'{label}\n(kHz/m)')
for ax in axes:
    for x, style in ((0.0, '-'), ((b - a) * 1e3, '-'), (tr.te_s * 1e3, '--')):
        ax.axvline(x, color='crimson' if style == '--' else '0.4', ls=style, lw=1.0)
axes[0].set_title('One RF-to-RF interval: solid = RF effective centres, dashed = echo')
axes[-1].set_xlabel('time from the RF effective centre (ms)')
fig.tight_layout()
plt.show()

`Gz` shows the pair plainly: a negative lobe before each pulse and another after it, with the
positive selection gradient between them. `Gy` is the blip and its mirror. `Gx` is the prephaser,
the readout, and the lobe that brings the axis back to zero.

## The scan: a start-up schedule and one loop

Nothing below is in the kernel. The phase alternates by 180° each repetition, which is the
standard bSSFP progression and puts the passband at the on-resonance frequency; the kernel takes
that value and gives it to the RF and the receiver together.

The start-up here is a linear flip-angle ramp. It is one of several schedules in use — a single
half-angle pulse half a TR early is another — and the simulation notebook measures what this one
does rather than claiming it is the best.

In [ ]:
STARTUP = 10          # ramp repetitions, sampling nothing
PHASE_STEP = 180.0    # the bSSFP alternation


def startup_kernels(n, final_deg=FLIP_DEG):
    """A linear flip-angle ramp into the steady state: n repetitions, none of them sampled."""
    return [
        sc.modules.bSSFP2DTR(
            opts=opts, fov_mm=FOV_MM, matrix=MATRIX, thickness_mm=THICKNESS_MM,
            flip_deg=final_deg * (k + 1) / (n + 1), bandwidth_hz_px=BANDWIDTH_HZ_PX,
            tr_s=tr.tr_s,
        )
        for k in range(n)
    ]


ramp = startup_kernels(STARTUP)
print('start-up ramp flip angles:',
      ', '.join(f'{k.exc.flip_deg:.1f}' for k in ramp), f'-> {FLIP_DEG:.1f} deg')
print('every repetition reports the same RF-centre offset:',
      len({round(k.time_to_rf_center(), 12) for k in [*ramp, tr]}) == 1)

That last line is worth a sentence. A repetition is placed at `start + TR` only if the next
repetition's RF effective centre sits at the same offset inside its block; otherwise the interval
the magnetisation sees is `block duration + c(n+1) - c(n)` and the declared TR is not what gets
played. A flip-angle ramp changes the RF amplitude and leaves the selection gradient and the
pulse envelope alone, so the offset does not move and simple stacking is exact. A ramp that also
changed the slice thickness or the pulse duration would not be; the reliable check is
`time_to_rf_center()` itself rather than the parameter that was varied.

### The continuous acquisition

The ramp, then every line in order. Four lines of policy.

In [ ]:
def continuous(kernel, lines, *, startup=STARTUP, tag='bssfp_2d'):
    """Start-up ramp, then one repetition per line, laid end to end."""
    out, at, n = sc.LogicBlock(tag), 0.0, 0
    for prep in startup_kernels(startup):
        out.add(at, prep(line=lines[0], phase_deg=PHASE_STEP * n, acquire=False))
        at, n = at + prep.tr_s, n + 1
    for line in lines:
        out.add(at, kernel(line=line, phase_deg=PHASE_STEP * n))
        at, n = at + kernel.tr_s, n + 1
    return out


lines = list(range(NY))
tree = continuous(tr, lines)
print(f'{STARTUP} start-up + {len(lines)} lines = {tree.duration * 1e3:.1f} ms '
      f'({tree.duration:.3f} s)')

In [ ]:
seq = sc.compile(tree, opts, name='bssfp_2d', definitions={
    'FOV': [FOV_MM * 1e-3, FOV_MM * 1e-3, THICKNESS_MM * 1e-3],
    'Matrix': [NX, NY, 1],
    'TE': tr.te_s, 'TR': tr.tr_s, 'FlipAngle': FLIP_DEG,
})
print(seq.duration()[0] * 1e3, 'ms total')

## What was actually encoded

The acquired lines come back out of the compiled sequence rather than out of the loop that wrote
them. `ky` at each echo is the line that was requested, and every echo arrives at the same TE.

The echo sample is where the readout crosses $k_x = 0$ to within a fraction of one k-space
step. It is not exactly zero, and the leftover is the same half-dwell that
`symmetry_residual_s` reports: the samples sit at the centres of their dwell intervals, so the
one nearest the crossing is up to half a dwell away from it. A reconstruction accounts for that
in the usual way, by using the sample times the sequence reports rather than assuming the centre
sample is the centre of k-space.

In [ ]:
k_adc, _, _, _, t_adc = seq.calculate_kspace()
samples = int(tr.ro.adc.num_samples)
per_tr = k_adc.reshape(3, -1, samples)
echo = tr.ro.echo_sample(0)

ky_at_echo = per_tr[1, :, echo]
kx_at_echo = per_tr[0, :, echo]
t_at_echo = t_adc.reshape(-1, samples)[:, echo]
rf_t = rf_centres(seq)[STARTUP:]
dk_x = 1.0 / (FOV_MM * 1e-3)

print(f'acquired repetitions   {per_tr.shape[1]}')
print(f'ky at echo matches the requested line  '
      f'{np.allclose(ky_at_echo, [tr.pe.k_per_m(line) for line in lines])}')
print(f'|kx| at echo           max {np.abs(kx_at_echo).max():.3f} 1/m  '
      f'= {np.abs(kx_at_echo).max() / dk_x:.3f} of one k-space step')
print(f'TE from each RF centre  {np.ptp(t_at_echo - rf_t) * 1e9:.1f} ns spread, '
      f'mean {np.mean(t_at_echo - rf_t) * 1e3:.4f} ms (declared {tr.te_s * 1e3:.4f})')

## Segmented k-space

A segmented acquisition splits the lines into groups and runs the start-up again at the top of
each group. It is what a cardiac cine does: one segment per heartbeat, the magnetisation left to
recover in between, the steady state re-established each time.

**This needs nothing from the kernel.** It is another loop around the same repetitions, so it is
written here, in the same four lines of policy with one more `for` around them.

In [ ]:
def segmented(kernel, lines, *, segments, startup=STARTUP, tag='bssfp_2d_segmented'):
    """`segments` groups of lines, each preceded by its own start-up ramp."""
    out, at, n = sc.LogicBlock(tag), 0.0, 0
    for segment in np.array_split(np.asarray(lines), segments):
        for prep in startup_kernels(startup):
            out.add(at, prep(line=int(segment[0]), phase_deg=PHASE_STEP * n, acquire=False))
            at, n = at + prep.tr_s, n + 1
        for line in segment:
            out.add(at, kernel(line=int(line), phase_deg=PHASE_STEP * n))
            at, n = at + kernel.tr_s, n + 1
    return out


SEGMENTS = 4
seg_tree = segmented(tr, lines, segments=SEGMENTS)
seg_seq = sc.compile(seg_tree, opts, name='bssfp_2d_segmented', definitions={
    'FOV': [FOV_MM * 1e-3, FOV_MM * 1e-3, THICKNESS_MM * 1e-3],
    'Matrix': [NX, NY, 1],
    'TE': tr.te_s, 'TR': tr.tr_s, 'FlipAngle': FLIP_DEG,
})

print(f'{SEGMENTS} segments of {NY // SEGMENTS} lines, each with {STARTUP} start-up repetitions')
print(f'continuous {tree.duration * 1e3:8.1f} ms')
print(f'segmented  {seg_tree.duration * 1e3:8.1f} ms  '
      f'(+{(seg_tree.duration / tree.duration - 1) * 100:.0f}%, all of it start-up)')

The segmented sequence is longer by exactly the extra start-up repetitions, and it encodes the
same lines. Balance is unaffected: it is a property of each RF-to-RF interval, and segmentation
changes which repetitions are adjacent, not what any of them plays.

Centre-ordering, a different segment count, interleaving, or skipping lines for parallel imaging
all go in the same place — they are edits to the two loops above.

In [ ]:
seg_k = seg_seq.calculate_kspace()[0].reshape(3, -1, samples)
print('segmented encodes the same lines as continuous:',
      np.allclose(np.sort(seg_k[1, :, echo]), np.sort(ky_at_echo)))

seg_centres = rf_centres(seg_seq)
worst = max(np.abs(m0_between(seg_seq, seg_centres[n], seg_centres[n + 1])).max()
            for n in range(len(seg_centres) - 1))
print(f'worst |M0| over any RF-to-RF interval, all {len(seg_centres) - 1} of them: '
      f'{worst:.2e} 1/m')

## The refusals

A repetition shorter than the hardware can play is refused at construction, with the achievable
value in the message. The two minima answer different questions: `min_tr_s` is the shortest
*symmetric* repetition, and `min_te_s` is the earliest the echo can arrive at all.

In [ ]:
for field, value in (('tr_s', tr.min_tr_s * 0.5), ('te_s', tr.min_te_s * 0.5)):
    try:
        sc.modules.bSSFP2DTR(
            opts=opts, fov_mm=FOV_MM, matrix=MATRIX, thickness_mm=THICKNESS_MM,
            flip_deg=FLIP_DEG, bandwidth_hz_px=BANDWIDTH_HZ_PX, **{field: value},
        )
    except sc.errors.ConfigurationError as err:
        print(f'--- {field}={value * 1e3:.3f} ms')
        print(str(err).strip(), end='\n\n')

## What the sequence does not say

The waveform is balanced from the first interval. Whether the **magnetisation** has reached a
steady state is a different question, answered by the tissue's $T_1$ and $T_2$, the flip angle,
TR, and how many repetitions have gone by — none of which a `.seq` file can settle.

So the start-up ramp above is a schedule, not a guarantee. The notebook beside this one simulates
the train and watches the transverse magnetisation settle, which is the only way to see whether
ten repetitions were enough for a given tissue.

## The files

In [ ]:
seq.write(str(SEQ_DIR / 'bssfp_2d.seq'))
seg_seq.write(str(SEQ_DIR / 'bssfp_2d_segmented.seq'))
np.savez(
    SEQ_DIR / 'bssfp_2d_nominal.npz',
    fov_mm=FOV_MM, matrix=np.array(MATRIX), thickness_mm=THICKNESS_MM,
    flip_deg=FLIP_DEG, bandwidth_hz_px=BANDWIDTH_HZ_PX,
    startup=STARTUP, segments=SEGMENTS, phase_step=PHASE_STEP,
    te_s=tr.te_s, tr_s=tr.tr_s, symmetry_residual_s=tr.symmetry_residual_s,
    time_to_rf_center_s=tr.time_to_rf_center(), center_line=tr.center_line,
    lines=np.array(lines), k_adc=per_tr,
)
print(f'-> {SEQ_DIR}/bssfp_2d.seq, bssfp_2d_segmented.seq, bssfp_2d_nominal.npz')

## Summary

- **Balanced** means zero net gradient area on every axis over the RF-to-RF interval, measured
  from one RF effective centre to the next. It is a property of the waveform, true from the first
  repetition.
- The z axis carries **two** balancing lobes, one either side of the pulse, because the interval
  of the previous repetition runs up to this pulse's centre and claims the first half of the
  selection gradient.
- `TE = TR/2` is the canonical symmetric realisation and the default. An explicit `te_s` gives an
  asymmetric bSSFP, which is still balanced.
- Every declared time is measured from the RF **effective** centre, which `time_to_rf_center()`
  reports; `time_to_echo() == time_to_rf_center() + te_s`.
- Repetitions can be stacked at `n * TR` when they agree on `time_to_rf_center()`, which a
  flip-angle ramp does not disturb.
- The start-up schedule, the line order and the segmentation are all written in this notebook.
  Segmentation needed no kernel parameter and no new class — one more `for` loop.
- A balanced waveform is not a magnetisation in steady state. That is what
  `02_simulate_and_reconstruct.ipynb` measures.

## References

1. Carr HY. *Steady-state free precession in nuclear magnetic resonance.* Phys Rev 1958;112:1693.
2. Oppelt A, Graumann R, Barfuss H, et al. *FISP — a new fast MRI sequence.* Electromedica
   1986;54:15.
3. Scheffler K, Lehnhardt S. *Principles and applications of balanced SSFP techniques.* Eur Radiol
   2003;13:2409.
4. Bieri O, Scheffler K. *Fundamentals of balanced steady state free precession MRI.* J Magn Reson
   Imaging 2013;38:2.
5. Bernstein MA, King KF, Zhou XJ. *Handbook of MRI Pulse Sequences.* Elsevier, 2004, §14.1.